# SQLarper's ETL Pipeline

In [32]:
import html
import json
import os
import random
import re
import xml.etree.ElementTree as ET
from datetime import datetime, timedelta, timezone
from pathlib import Path

import polars as pl
from dotenv import find_dotenv, load_dotenv
from sqlalchemy import create_engine, text

# Which database to load into: "TEST" or "PROD" 
TARGET = "TEST"
TEST_DB_NAME = "__catcruxis"


RAW_DIR = Path("../data/raw")
WILDCATSYNC_FILE = RAW_DIR / "wildcatsync.xml"
HURT_HUB_FILE = RAW_DIR / "hurthub.json"
DAVIDSON_FILE = RAW_DIR / "davidson.json"

# davidson.edu gives local times without a time zone
LOCAL_TZ = "America/New_York"

# When scraping.py downloaded the raw files. Update this if you rerun scraping.py.
DATE_FOUND = datetime(2026, 11, 4, tzinfo=timezone.utc)

load_dotenv(find_dotenv(usecwd=True))  

if TARGET == "TEST":
    db_url = f"{os.environ['DEV_DB_SERVER']}/{TEST_DB_NAME}"
elif TARGET == "PROD":
    db_url = os.environ["PROD_DB_URL"]
else:
    raise ValueError("TARGET must be 'TEST' or 'PROD'")

db_url = db_url.replace("postgres://", "postgresql://", 1)
engine = create_engine(db_url)

## 2. Extract


### 2a WildcatSync

In [33]:
EV = "{events}"   # the namespace WildcatSync uses for start, end, location, status, host

items = ET.parse(WILDCATSYNC_FILE).getroot().findall(".//item")

ws_raw = pl.DataFrame(
    [
        {
            "title": it.findtext("title"),
            "link": it.findtext("link"),
            "description_html": it.findtext("description"),
            "category": it.findtext("category"),
            "start": it.findtext(EV + "start"),
            "end": it.findtext(EV + "end"),
            "location": it.findtext(EV + "location"),
            "status": it.findtext(EV + "status"),
            "hosts": [h.text for h in it.findall(EV + "host")],
        }
        for it in items
    ],
    schema_overrides={"hosts": pl.List(pl.String)},
)
print(ws_raw.shape)
ws_raw.head(3)

(106, 9)


title,link,description_html,category,start,end,location,status,hosts
str,str,str,str,str,str,str,str,list[str]
"""Social Gathering ""","""https://wildcatsync.davidson.e…","""<div class=""h-event vevent""> …","""Social""","""Wed, 04 Nov 2026 23:30:00 GMT""","""Thu, 05 Nov 2026 00:30:00 GMT""","""Watson 2nd Floor Lounge""","""confirmed""","[""Davidson College ACM Student Chapter""]"
"""Wednesday Night Zen""","""https://wildcatsync.davidson.e…","""<div class=""h-event vevent""> …","""Spirituality""","""Thu, 05 Nov 2026 00:00:00 GMT""","""Thu, 05 Nov 2026 01:00:00 GMT""","""Oasis""","""confirmed""","[""Buddhism at Davidson""]"
"""Mewing, Mogging, and Mental He…","""https://wildcatsync.davidson.e…","""<div class=""h-event vevent""> …","""ThoughtfulLearning""","""Thu, 05 Nov 2026 00:00:00 GMT""","""Thu, 05 Nov 2026 01:00:00 GMT""","""900 Room""","""confirmed""","[""Center for Student Health & Well-Being""]"


### 2b. Hurt Hub 

In [34]:
hh_json = json.loads(HURT_HUB_FILE.read_text(encoding="utf-8"))["events"]

hh_raw = pl.DataFrame(
    [
        {
            "title": e["title"],
            "url": e["url"],
            "description_html": e["description"],
            "utc_start_date": e["utc_start_date"],
            "utc_end_date": e["utc_end_date"],
            "venue": e["venue"].get("venue") if isinstance(e["venue"], dict) else None,
            "organizers": [o["organizer"] for o in e["organizer"]],
            "categories": [c["name"] for c in e["categories"]],
        }
        for e in hh_json
    ],
    schema_overrides={"organizers": pl.List(pl.String), "categories": pl.List(pl.String)},
)
print(hh_raw.shape)
hh_raw

(6, 8)


title,url,description_html,utc_start_date,utc_end_date,venue,organizers,categories
str,str,str,str,str,str,list[str],list[str]
"""Building a Lean Startup &#8211…","""https://hurthub.davidson.edu/e…","""<div>Learn how to jumpstart yo…","""2026-09-23 22:45:00""","""2026-11-12 01:15:00""","""The Hurt Hub at Davidson Colle…","[""The Hurt Hub@Davidson""]","[""Class"", ""Discussion""]"
"""The Extended Mind: A Field Gui…","""https://hurthub.davidson.edu/e…","""<div>Facilitated by Peter Gray…","""2026-11-04 21:30:00""","""2026-11-04 22:30:00""","""The Hurt Hub at Davidson Colle…","[""The Hurt Hub@Davidson""]",[]
"""Shop Local at the Makers&#8217…","""https://hurthub.davidson.edu/e…","""<p>The holidays are just aroun…","""2026-11-13 21:00:00""","""2026-11-14 00:00:00""","""The Hurt Hub at Davidson Colle…",[],"[""Social""]"
"""The Hurt Hub Innovation &amp; …","""https://hurthub.davidson.edu/e…","""<div>The Hurt Hub Innovation &…","""2026-11-18 23:00:00""","""2026-11-19 02:00:00""","""The Hurt Hub at Davidson Colle…","[""The Hurt Hub@Davidson""]",[]
"""CPA Office Hours (SCHWINN CPA …","""https://hurthub.davidson.edu/e…","""<p>The Hurt Hub has partnered …","""2026-11-19 14:30:00""","""2026-11-19 17:30:00""",null,[],"[""Amplify Davidson"", ""Discussion"", … ""Virtual""]"
"""Solopreneur Circle""","""https://hurthub.davidson.edu/e…","""<div>A monthly community gathe…","""2026-11-20 14:00:00""","""2026-11-20 15:00:00""","""The Hurt Hub at Davidson Colle…","[""The Hurt Hub@Davidson""]",[]


### 2c. Davidson College Official Events

In [35]:
dv_json = json.loads(DAVIDSON_FILE.read_text(encoding="utf-8"))["events"]

dv_raw = pl.DataFrame(
    [
        {k: e.get(k) for k in ("title", "url", "description", "start", "end", "location", "date_time_text")}
        for e in dv_json
    ]
)
print(dv_raw.shape)
dv_raw.head(3)

(45, 7)


title,url,description,start,end,location,date_time_text
str,str,str,str,str,str,str
"""Charif Shanahan Poetry Reading""","""https://www.davidson.edu/event…","""November 04 5:00PM - 6:30PM …","""2026-11-04T17:00:00""","""2026-11-04T18:30:00""","""UNION 307 - Smith 900 Room""","""November 04 5:00PM - 6:30PM"""
"""Solar Canopy Dedication""","""https://www.davidson.edu/event…","""November 05 11:05AM - 12:00PM…","""2026-11-05T11:05:00""","""2026-11-05T12:00:00""","""Baker Sports Complex South Law…","""November 05 11:05AM - 12:00PM"""
"""Piano Studio Fall Recital""","""https://www.davidson.edu/event…","""November 05 7:30PM - 9:00PM …","""2026-11-05T19:30:00""","""2026-11-05T21:00:00""","""SLOAN TY-TL - Tyler-Tallman Re…","""November 05 7:30PM - 9:00PM"""


### 2d. Profiling
Before fixing anything, we look at what we have: missing values, multi-valued cells, formats and repeats. Each finding becomes a cleaning step in section 3 (labelled F1, F2, ...).

In [36]:
for name, df in (("WildcatSync", ws_raw), ("Hurt Hub", hh_raw), ("Davidson", dv_raw)):
    print(name, df.null_count().to_dicts()[0])

WildcatSync {'title': 0, 'link': 0, 'description_html': 0, 'category': 0, 'start': 0, 'end': 0, 'location': 0, 'status': 0, 'hosts': 0}
Hurt Hub {'title': 0, 'url': 0, 'description_html': 0, 'utc_start_date': 0, 'utc_end_date': 0, 'venue': 1, 'organizers': 0, 'categories': 0}
Davidson {'title': 0, 'url': 0, 'description': 0, 'start': 0, 'end': 0, 'location': 0, 'date_time_text': 0}


Cancelled events and hosts per event:



In [37]:
print(ws_raw["status"].value_counts())
print(ws_raw.select(pl.col("hosts").list.len().alias("hosts_per_event"))["hosts_per_event"].value_counts().sort("hosts_per_event"))

shape: (2, 2)
┌───────────┬───────┐
│ status    ┆ count │
│ ---       ┆ ---   │
│ str       ┆ u32   │
╞═══════════╪═══════╡
│ confirmed ┆ 104   │
│ cancelled ┆ 2     │
└───────────┴───────┘
shape: (6, 2)
┌─────────────────┬───────┐
│ hosts_per_event ┆ count │
│ ---             ┆ ---   │
│ u32             ┆ u32   │
╞═════════════════╪═══════╡
│ 1               ┆ 61    │
│ 2               ┆ 36    │
│ 3               ┆ 4     │
│ 4               ┆ 2     │
│ 5               ┆ 2     │
│ 6               ┆ 1     │
└─────────────────┴───────┘


Categories:


In [38]:
print(ws_raw["category"].value_counts().sort("count", descending=True))
print(hh_raw.select(pl.col("categories").explode())["categories"].value_counts())

shape: (9, 2)
┌────────────────────┬───────┐
│ category           ┆ count │
│ ---                ┆ ---   │
│ str                ┆ u32   │
╞════════════════════╪═══════╡
│ ThoughtfulLearning ┆ 42    │
│ Spirituality       ┆ 16    │
│ Athletics          ┆ 15    │
│ Social             ┆ 15    │
│ Cultural           ┆ 8     │
│ CommunityService   ┆ 4     │
│ Fundraising        ┆ 3     │
│ Arts               ┆ 2     │
│ GroupBusiness      ┆ 1     │
└────────────────────┴───────┘
shape: (8, 2)
┌───────────────────┬───────┐
│ categories        ┆ count │
│ ---               ┆ ---   │
│ str               ┆ u32   │
╞═══════════════════╪═══════╡
│ Class             ┆ 1     │
│ Discussion        ┆ 2     │
│ Social            ┆ 1     │
│ Office Hours      ┆ 1     │
│ Free for Students ┆ 1     │
│ Amplify Davidson  ┆ 1     │
│ null              ┆ 3     │
│ Virtual           ┆ 1     │
└───────────────────┴───────┘


/tmp/ipykernel_2980467/1681116005.py:2: DeprecationWarning: In Polars 2.0, the default behavior for `empty_as_null` will change to `False`. To keep the current behavior, explicitly set `empty_as_null=True`.
  print(hh_raw.select(pl.col("categories").explode())["categories"].value_counts())


Date formats:


In [39]:
print("WildcatSync:", ws_raw["start"][0])
print("Hurt Hub:   ", hh_raw["utc_start_date"][0])
print("Davidson:   ", dv_raw["start"][0])

WildcatSync: Wed, 04 Nov 2026 23:30:00 GMT
Hurt Hub:    2026-09-23 22:45:00
Davidson:    2026-11-04T17:00:00


Location text:



In [40]:
all_locations = pl.concat([
    ws_raw.select("location"),
    hh_raw.select(pl.col("venue").alias("location")),
    dv_raw.select("location"),
])
print("distinct location strings:", all_locations["location"].n_unique())
all_locations["location"].value_counts().sort("count", descending=True).head(25)

distinct location strings: 81


location,count
str,u32
"""SLOAN TY-TL - Tyler-Tallman Re…",19
"""Cannon Pool - Baker Sports Com…",11
"""Duke Family Performance Hall""",8
"""Oasis""",8
"""The Hurt Hub at Davidson Colle…",5
…,…
"""Hance Auditorium, Chambers Bui…",2
"""Lilly Family Gallery""",2
"""https://www.faoschwarzfellowsh…",1


Messy titles:



In [41]:
print(ws_raw.filter(pl.col("title") != pl.col("title").str.strip_chars()).select("title").head())
print(hh_raw.filter(pl.col("title").str.contains("&")).select("title"))

shape: (5, 1)
┌─────────────────────────────────┐
│ title                           │
│ ---                             │
│ str                             │
╞═════════════════════════════════╡
│ Social Gathering                │
│ Mindful Yoga                    │
│ A Conversation About Vaccines … │
│ Cats Excursions: Murder on the… │
│  Yearbook Meeting  ✏️           │
└─────────────────────────────────┘
shape: (3, 1)
┌─────────────────────────────────┐
│ title                           │
│ ---                             │
│ str                             │
╞═════════════════════════════════╡
│ Building a Lean Startup &#8211… │
│ Shop Local at the Makers&#8217… │
│ The Hurt Hub Innovation &amp; … │
└─────────────────────────────────┘


The same event on two sources, or posted twice:



In [42]:
def title_key(col):
    return pl.col(col).str.to_lowercase().str.replace_all(r"[^a-z0-9]", "")

ws_keys = ws_raw.select(title_key("title").alias("k"), "start")
dv_keys = dv_raw.select(title_key("title").alias("k"))
print("on both WildcatSync and davidson.edu:")
print(ws_keys.join(dv_keys, on="k", how="inner").select("k").unique())
print("posted twice on WildcatSync (same title and start):")
print(ws_keys.filter(pl.struct("k", "start").is_duplicated()))

on both WildcatSync and davidson.edu:
shape: (2, 1)
┌─────────────────────────────────┐
│ k                               │
│ ---                             │
│ str                             │
╞═════════════════════════════════╡
│ remembranceandreconnectionacom… │
│ solarcanopydedication           │
└─────────────────────────────────┘
posted twice on WildcatSync (same title and start):
shape: (4, 2)
┌─────────────────────────────┬───────────────────────────────┐
│ k                           ┆ start                         │
│ ---                         ┆ ---                           │
│ str                         ┆ str                           │
╞═════════════════════════════╪═══════════════════════════════╡
│ bonnerscholarsmondaymeeting ┆ Mon, 16 Nov 2026 22:00:00 GMT │
│ bonnerscholarsmondaymeeting ┆ Mon, 16 Nov 2026 22:00:00 GMT │
│ largegroup                  ┆ Wed, 18 Nov 2026 00:30:00 GMT │
│ largegroup                  ┆ Wed, 18 Nov 2026 00:30:00 GMT │
└─────────────────

## 3. Transform: clean

Each step fixes one finding from the profiling (F1–F11). First, two helpers that all three sources use.

In [43]:
def strip_html(value):
    # F2/F3: remove tags, decode codes like &amp; and squeeze whitespace
    if value is None:
        return None
    no_tags = re.sub(r"<[^>]+>", " ", value)
    cleaned = " ".join(html.unescape(no_tags).split())
    return cleaned or None


def clean_text(expr):
    # strip spaces at both ends and collapse runs of spaces inside
    return expr.str.strip_chars().str.replace_all(r"\s+", " ")

### 3a. WildcatSync and “F1 drop cancelled; F2/F3 clean title and description (keep only the p-description part of the HTML card); F4 parse GMT dates as UTC; F6 split CamelCase categories.”

In [44]:
def ws_description(card_html):
    # The HTML card has a <div class="p-description ..."> holding the real text
    match = re.search(r'class="p-description[^"]*">(.*?)</div>', card_html or "", re.S)
    return strip_html(match.group(1)) if match else None


ws = (
    ws_raw
    .filter(pl.col("status") == "confirmed")                                        # F1
    .with_columns(
        clean_text(pl.col("title")).alias("title"),                                 # F2
        pl.col("description_html").map_elements(ws_description, return_dtype=pl.String).alias("description"),  # F3
        pl.col("start").str.to_datetime("%a, %d %b %Y %H:%M:%S GMT", time_zone="UTC").alias("start_time"),     # F4
        pl.col("end").str.to_datetime("%a, %d %b %Y %H:%M:%S GMT", time_zone="UTC").alias("end_time"),
        pl.col("category").str.replace_all(r"([a-z])([A-Z])", "${1} ${2}")          # F6
            .map_elements(lambda c: [c], return_dtype=pl.List(pl.String)).alias("categories"),
        pl.col("hosts").alias("organizers"),
        pl.col("link").alias("source_url"),
        pl.lit("WildcatSync").alias("source_name"),
    )
    .select("title", "description", "start_time", "end_time", "location",
            "organizers", "categories", "source_name", "source_url")
)
print(ws.shape)
ws.head(3)

(104, 9)


title,description,start_time,end_time,location,organizers,categories,source_name,source_url
str,str,"datetime[μs, UTC]","datetime[μs, UTC]",str,list[str],list[str],str,str
"""Social Gathering""","""Hang out with fellow CS studen…",2026-11-04 23:30:00 UTC,2026-11-05 00:30:00 UTC,"""Watson 2nd Floor Lounge""","[""Davidson College ACM Student Chapter""]","[""Social""]","""WildcatSync""","""https://wildcatsync.davidson.e…"
"""Wednesday Night Zen""","""Interested in Meditation? In Z…",2026-11-05 00:00:00 UTC,2026-11-05 01:00:00 UTC,"""Oasis""","[""Buddhism at Davidson""]","[""Spirituality""]","""WildcatSync""","""https://wildcatsync.davidson.e…"
"""Mewing, Mogging, and Mental He…","""Join us for ""Mewing, Mogging, …",2026-11-05 00:00:00 UTC,2026-11-05 01:00:00 UTC,"""900 Room""","[""Center for Student Health & Well-Being""]","[""Thoughtful Learning""]","""WildcatSync""","""https://wildcatsync.davidson.e…"


### 3b. Hurt Hub and “F2/F3 decode HTML in titles and descriptions; F4 the utc_* fields are already UTC, so we just mark them as UTC.”

In [45]:
hh = (
    hh_raw
    .with_columns(
        pl.col("title").map_elements(strip_html, return_dtype=pl.String).alias("title"),                   # F2
        pl.col("description_html").map_elements(strip_html, return_dtype=pl.String).alias("description"),  # F3
        pl.col("utc_start_date").str.to_datetime("%Y-%m-%d %H:%M:%S", time_zone="UTC").alias("start_time"),  # F4
        pl.col("utc_end_date").str.to_datetime("%Y-%m-%d %H:%M:%S", time_zone="UTC").alias("end_time"),
        pl.col("venue").alias("location"),
        pl.col("url").alias("source_url"),
        pl.lit("Hurt Hub").alias("source_name"),
    )
    .select("title", "description", "start_time", "end_time", "location",
            "organizers", "categories", "source_name", "source_url")
)
hh

title,description,start_time,end_time,location,organizers,categories,source_name,source_url
str,str,"datetime[μs, UTC]","datetime[μs, UTC]",str,list[str],list[str],str,str
"""Building a Lean Startup – Fall…","""Learn how to jumpstart your bu…",2026-09-23 22:45:00 UTC,2026-11-12 01:15:00 UTC,"""The Hurt Hub at Davidson Colle…","[""The Hurt Hub@Davidson""]","[""Class"", ""Discussion""]","""Hurt Hub""","""https://hurthub.davidson.edu/e…"
"""The Extended Mind: A Field Gui…","""Facilitated by Peter Gray, PhD…",2026-11-04 21:30:00 UTC,2026-11-04 22:30:00 UTC,"""The Hurt Hub at Davidson Colle…","[""The Hurt Hub@Davidson""]",[],"""Hurt Hub""","""https://hurthub.davidson.edu/e…"
"""Shop Local at the Makers’ Mark…","""The holidays are just around t…",2026-11-13 21:00:00 UTC,2026-11-14 00:00:00 UTC,"""The Hurt Hub at Davidson Colle…",[],"[""Social""]","""Hurt Hub""","""https://hurthub.davidson.edu/e…"
"""The Hurt Hub Innovation & Entr…","""The Hurt Hub Innovation & Entr…",2026-11-18 23:00:00 UTC,2026-11-19 02:00:00 UTC,"""The Hurt Hub at Davidson Colle…","[""The Hurt Hub@Davidson""]",[],"""Hurt Hub""","""https://hurthub.davidson.edu/e…"
"""CPA Office Hours (SCHWINN CPA …","""The Hurt Hub has partnered wit…",2026-11-19 14:30:00 UTC,2026-11-19 17:30:00 UTC,null,[],"[""Amplify Davidson"", ""Discussion"", … ""Virtual""]","""Hurt Hub""","""https://hurthub.davidson.edu/e…"
"""Solopreneur Circle""","""A monthly community gathering …",2026-11-20 14:00:00 UTC,2026-11-20 15:00:00 UTC,"""The Hurt Hub at Davidson Colle…","[""The Hurt Hub@Davidson""]",[],"""Hurt Hub""","""https://hurthub.davidson.edu/e…"


### 3c. Davidson College Events and “F3 drop the date/time lines repeated at the start of the description; F4 times are local, so we say they’re America/New_York, then convert to UTC; F7 no organizers or categories, so empty lists for now.”

In [46]:
def dv_description(desc, date_time_text):
    # F3: drop leading paragraphs that just repeat the date and time
    if desc is None:
        return None
    paragraphs = desc.split("\n\n")
    # e.g. "November 04 5:00PM - 6:30PM" repeats as "November 04" then "5:00PM - 6:30PM"
    while paragraphs and paragraphs[0] in (date_time_text or ""):
        paragraphs.pop(0)
    return strip_html("\n\n".join(paragraphs))


dv = (
    dv_raw
    .with_columns(
        clean_text(pl.col("title")).alias("title"),
        pl.struct("description", "date_time_text").map_elements(
            lambda r: dv_description(r["description"], r["date_time_text"]), return_dtype=pl.String
        ).alias("description"),                                                       # F3
        pl.col("start").str.to_datetime("%Y-%m-%dT%H:%M:%S")                          # F4
            .dt.replace_time_zone(LOCAL_TZ).dt.convert_time_zone("UTC").alias("start_time"),
        pl.col("end").str.to_datetime("%Y-%m-%dT%H:%M:%S")
            .dt.replace_time_zone(LOCAL_TZ).dt.convert_time_zone("UTC").alias("end_time"),
        pl.lit([], dtype=pl.List(pl.String)).alias("organizers"),                    # F7
        pl.lit([], dtype=pl.List(pl.String)).alias("categories"),
        pl.col("url").alias("source_url"),
        pl.lit("Davidson College Events").alias("source_name"),
    )
    .select("title", "description", "start_time", "end_time", "location",
            "organizers", "categories", "source_name", "source_url")
)
dv.head(3)

title,description,start_time,end_time,location,organizers,categories,source_name,source_url
str,str,"datetime[μs, UTC]","datetime[μs, UTC]",str,list[str],list[str],str,str
"""Charif Shanahan Poetry Reading""","""Poet Charif Shanahan will read…",2026-11-04 22:00:00 UTC,2026-11-04 23:30:00 UTC,"""UNION 307 - Smith 900 Room""",[],[],"""Davidson College Events""","""https://www.davidson.edu/event…"
"""Solar Canopy Dedication""","""Join us on November 5 for the …",2026-11-05 16:05:00 UTC,2026-11-05 17:00:00 UTC,"""Baker Sports Complex South Law…",[],[],"""Davidson College Events""","""https://www.davidson.edu/event…"
"""Piano Studio Fall Recital""","""Students from the studios of C…",2026-11-06 00:30:00 UTC,2026-11-06 02:00:00 UTC,"""SLOAN TY-TL - Tyler-Tallman Re…",[],[],"""Davidson College Events""","""https://www.davidson.edu/event…"


### 3d. Combine the three sources and “Same columns and types, so we stack them: one row per (event, source).”

In [47]:
cleaned = pl.concat([ws, hh, dv])
print(cleaned.shape)
cleaned.group_by("source_name").len()

(155, 9)


source_name,len
str,u32
"""Hurt Hub""",6
"""Davidson College Events""",45
"""WildcatSync""",104


### 3e. Organizers and categories (F7, F8)
- F8: map known spellings of the same organizer to one name.
- F7: if an event has no organizer, use the source's own organizer. If it has no category, use `General`.


In [48]:
# F8: different spellings of the same organizer -> one name
ORGANIZER_ALIASES = {
    "Hurt Hub@Davidson": "The Hurt Hub",
    "The Hurt Hub@Davidson": "The Hurt Hub",
}

# F7: who runs events that list no organizer
FALLBACK_ORGANIZER = {
    "WildcatSync": "Davidson College",
    "Hurt Hub": "The Hurt Hub",
    "Davidson College Events": "Davidson College",
}
FALLBACK_CATEGORY = "General"

cleaned = cleaned.with_columns(
    pl.col("organizers").list.eval(clean_text(pl.element()).replace(ORGANIZER_ALIASES)),
    pl.col("categories").list.eval(clean_text(pl.element())),
).with_columns(
    pl.when(pl.col("organizers").list.len() == 0)
      .then(pl.col("source_name").replace_strict(FALLBACK_ORGANIZER)
            .map_elements(lambda o: [o], return_dtype=pl.List(pl.String)))
      .otherwise(pl.col("organizers"))
      .alias("organizers"),
    pl.when(pl.col("categories").list.len() == 0)
      .then(pl.lit([FALLBACK_CATEGORY], dtype=pl.List(pl.String)))
      .otherwise(pl.col("categories"))
      .alias("categories"),
)
print("events with no organizer:", cleaned.filter(pl.col("organizers").list.len() == 0).height)
print("events with no category: ", cleaned.filter(pl.col("categories").list.len() == 0).height)

events with no organizer: 0
events with no category:  0


### 3f. Locations: building + room (F9)
1. Collect every distinct location string, so each rule runs once per string.
2. Map each string to (building_name, room_number) with a rule table: "if the text contains this word, the building is X". The room is a room number or a named spot.
3. URLs, Zoom links and empty values go to `Online / TBD`. Anything no rule matches keeps its cleaned text as the building, with no room.

We only merge spellings when the text itself names the building. To fix a location later, add a rule here.

In [49]:
ONLINE = "Online / TBD"

# (word to look for, building name). First match wins, so specific rules go first.
BUILDING_RULES = [
    ("hurt hub", "The Hurt Hub"),
    ("cannon pool", "Baker Sports Complex"),
    ("baker", "Baker Sports Complex"),
    ("oasis", "Alvarez College Union"),
    ("900 room", "Alvarez College Union"),
    ("morcott", "Alvarez College Union"),
    ("union", "Alvarez College Union"),
    ("alvarez", "Alvarez College Union"),
    ("chambers", "Chambers Building"),
    ("hance", "Chambers Building"),
    ("baxter-davidson", "Chambers Building"),
    ("wall", "Wall Academic Center"),
    ("sloan", "Sloan Music Center"),
    ("tyler-tallman", "Sloan Music Center"),
    ("cunn ", "Cunningham Fine Arts Building"),
    ("barber theatre", "Cunningham Fine Arts Building"),
    ("visual arts center", "Visual Arts Center"),
    ("vac ", "Visual Arts Center"),
    ("spencer", "Spencer-Weinstein Center"),
    ("with these hands", "With These Hands Memorial"),
    ("dcpc", "Davidson College Presbyterian Church"),
    ("watson", "Watson Life Sciences Building"),
]

# Named spots inside a building (used when there's no room number)
ROOM_NAMES = [
    "Hance Auditorium", "Baxter-Davidson Room", "Tyler-Tallman Recital Hall", "Barber Theatre",
    "Semans Lecture Hall", "Smith 900 Room", "900 Room", "Morcott Room", "Oasis", "Atrium",
    "Amphitheater", "Ampitheater", "Cannon Pool", "Dance Studio A", "South Lawn", "Parking Lot",
    "Flagpole", "2nd Floor Lounge", "Hamilton W. McKay Atrium",
]


def split_location(raw):
    # F9: turn one free-text location into (building_name, room_number)
    text_ = " ".join((raw or "").split())
    lower = text_.lower()
    if not text_ or lower.startswith("http") or "zoom" in lower or "virtual" in lower:
        return ONLINE, None

    building = next((b for word, b in BUILDING_RULES if word in lower + " "), None)
    if building is None:
        return text_, None            # unknown place: keep the cleaned text, no room

    number = re.search(r"\b(\d{3,4})\b", text_)
    if number:
        return building, number.group(1)
    for name in ROOM_NAMES:
        if name.lower() in lower:
            return building, "Amphitheater" if name == "Ampitheater" else name   # fix one misspelling
    return building, None             # the building in general


# Run the rules once per distinct location ("" stands for a missing location)
raw_locations = cleaned["location"].fill_null("").unique().sort().to_list()
location_map = pl.DataFrame(
    [{"raw_location": raw, "building_name": b, "room_number": r}
     for raw in raw_locations
     for b, r in [split_location(raw)]],
    schema={"raw_location": pl.String, "building_name": pl.String, "room_number": pl.String},
)
print("distinct raw locations:", location_map.height)
print("distinct buildings:    ", location_map["building_name"].n_unique())
location_map.sort("building_name")

distinct raw locations: 80
distinct buildings:     24


raw_location,building_name,room_number
str,str,str
"""900 Room""","""Alvarez College Union""","""900"""
"""900 Room in Union""","""Alvarez College Union""","""900"""
"""Alvarez College Union ""","""Alvarez College Union""",null
"""Alvarez College Union Atrium""","""Alvarez College Union""","""Atrium"""
"""C Shaw Smith 900 Room and vari…","""Alvarez College Union""","""900"""
…,…,…
"""Wall 320""","""Wall Academic Center""","""320"""
"""Wall 320 Classroom""","""Wall Academic Center""","""320"""
"""Watson 2nd Floor Lounge""","""Watson Life Sciences Building""","""2nd Floor Lounge"""


In [50]:
# Attach each event's building and room using the map above
cleaned = (
    cleaned.with_columns(pl.col("location").fill_null("").alias("raw_location"))
    .join(location_map, on="raw_location", how="left")
    .drop("raw_location")
)
print("events without a building:", cleaned.filter(pl.col("building_name").is_null()).height)

events without a building: 0


### 3g. Official status
No source has `official_status`, so we set it with a rule:
- davidson.edu and Hurt Hub events are posted by the college: `official`.
- A WildcatSync event is `official` if any host looks like a college office (name contains Office, Center, Program, Department...). Otherwise `student_organized`, since most WildcatSync hosts are student clubs.

In [51]:
OFFICIAL_WORDS = ["office", "center", "program", "department", "institute", "library", "division",
                  "dean", "galleries", "education abroad", "residence life", "student engagement",
                  "four year advising", "hurt hub", "davidson college"]
official_pattern = "|".join(re.escape(w) for w in OFFICIAL_WORDS)

cleaned = cleaned.with_columns(
    pl.when(pl.col("source_name") != "WildcatSync")
      .then(pl.lit("official"))
      .when(pl.col("organizers").list.eval(
          pl.element().str.to_lowercase().str.contains(official_pattern)).list.any())
      .then(pl.lit("official"))
      .otherwise(pl.lit("student_organized"))
      .alias("official_status")
)
cleaned.group_by("source_name", "official_status").len().sort("source_name")

source_name,official_status,len
str,str,u32
"""Davidson College Events""","""official""",45
"""Hurt Hub""","""official""",6
"""WildcatSync""","""student_organized""",41
"""WildcatSync""","""official""",63


### 3h. Duplicates: one event, many sources (F10)
An event is identified by its title (lowercase, letters and digits only) plus its start time. Rows that share this key are the same event. In Normalize we keep one row of event details (from the highest-priority source: WildcatSync, then Hurt Hub, then Davidson) and one `event_source` row for every source.

In [52]:
SOURCE_PRIORITY = {"WildcatSync": 1, "Hurt Hub": 2, "Davidson College Events": 3}

cleaned = cleaned.with_columns(
    (pl.col("title").str.to_lowercase().str.replace_all(r"[^a-z0-9]", "")
     + "|" + pl.col("start_time").dt.strftime("%Y-%m-%dT%H:%M")).alias("event_key"),
    pl.col("source_name").replace_strict(SOURCE_PRIORITY, return_dtype=pl.Int64).alias("priority"),
)
dupes = cleaned.filter(pl.col("event_key").is_duplicated()).select("title", "source_name", "start_time")
print("rows that share an event with another row:")
dupes.sort("title")

rows that share an event with another row:


title,source_name,start_time
str,str,"datetime[μs, UTC]"
"""Bonner Scholars Monday Meeting""","""WildcatSync""",2026-11-16 22:00:00 UTC
"""Bonner Scholars Monday Meeting""","""WildcatSync""",2026-11-16 22:00:00 UTC
"""Large Group""","""WildcatSync""",2026-11-18 00:30:00 UTC
"""Large Group""","""WildcatSync""",2026-11-18 00:30:00 UTC
"""Remembrance and Reconnection: …","""WildcatSync""",2026-11-06 22:00:00 UTC
"""Remembrance and Reconnection: …","""Davidson College Events""",2026-11-06 22:00:00 UTC
"""Solar Canopy Dedication""","""WildcatSync""",2026-11-05 16:05:00 UTC
"""Solar Canopy Dedication""","""Davidson College Events""",2026-11-05 16:05:00 UTC


In [53]:
# Sanity checks on the clean data. Each one fails loudly if something is off.
assert cleaned.height == ws_raw.filter(pl.col("status") == "confirmed").height + hh_raw.height + dv_raw.height, "rows lost or added"
assert cleaned["title"].null_count() == 0, "an event has no title"
assert cleaned["start_time"].null_count() == 0, "an event has no start time"
assert cleaned.filter(pl.col("end_time") < pl.col("start_time")).height == 0, "an event ends before it starts"
assert not cleaned["title"].str.contains("&#|&amp;").any(), "HTML codes left in a title"
assert not cleaned["description"].str.contains("<[a-z/]").fill_null(False).any(), "HTML tags left in a description"
assert cleaned["building_name"].null_count() == 0, "an event has no building"
assert (cleaned["organizers"].list.len() > 0).all(), "an event has no organizer"
assert (cleaned["categories"].list.len() > 0).all(), "an event has no category"
print("all clean-data checks passed")

all clean-data checks passed


## 4. Transform: normalize

We split the clean data into one DataFrame per table, with the same column names as the database (without `created_at`/`modified_at`, which the database fills in).
We give every row its primary key here, so foreign keys can be checked in Polars before anything touches the database. `source` is different: its rows come from the reference rows migration, so we read its IDs from the database.

In [54]:
with engine.connect() as conn:
    source = pl.read_database("SELECT source_id, source_name, source_type FROM source ORDER BY source_id", conn)
all_rows = cleaned   # every (event, source) row
source

source_id,source_name,source_type
i64,str,str
1,"""WildcatSync""","""platform"""
2,"""Hurt Hub""","""website"""
3,"""Davidson College Events""","""website"""


4a. building and location

In [55]:
building = (
    all_rows.select("building_name").unique().sort("building_name")
    .with_row_index("building_id", offset=1)
    .with_columns(
        pl.col("building_id").cast(pl.Int32),
        # No source has coordinates, so they stay empty for now
        pl.lit(None, dtype=pl.Decimal(9, 6)).alias("latitude"),
        pl.lit(None, dtype=pl.Decimal(9, 6)).alias("longitude"),
    )
)

# A text key for (building, room). Joining on it avoids comparing NULL rooms.
def location_key():
    return pl.col("building_name") + "|" + pl.col("room_number").fill_null("")

all_rows = all_rows.with_columns(location_key().alias("location_key"))

location = (
    all_rows.select("building_name", "room_number", "location_key").unique()
    .join(building.select("building_id", "building_name"), on="building_name")
    .sort("building_name", "room_number", nulls_last=True)
    .with_row_index("location_id", offset=1)
    .with_columns(pl.col("location_id").cast(pl.Int32))
)
print(building.height, "buildings,", location.height, "locations")

24 buildings, 46 locations


4b. “One row per event_key. When an event came from several sources, its details come from the highest-priority source.”

In [56]:
event_rows = (
    all_rows.sort("priority")
    .unique(subset="event_key", keep="first", maintain_order=True)
    .sort("start_time", "title")
    .with_row_index("event_id", offset=1)
    .with_columns(pl.col("event_id").cast(pl.Int32))
    .join(location.select("location_id", "location_key"), on="location_key", how="left")
)
event = event_rows.select("event_id", "location_id", "title", "description",
                          "start_time", "end_time", "official_status")

# Every (event, source) row with its event_id. The bridge tables use this.
rows_with_ids = all_rows.join(event_rows.select("event_key", "event_id"), on="event_key")
print(event.height, "events from", all_rows.height, "source rows")
event.head()

151 events from 155 source rows


event_id,location_id,title,description,start_time,end_time,official_status
i32,i32,str,str,"datetime[μs, UTC]","datetime[μs, UTC]",str
1,36,"""Building a Lean Startup – Fall…","""Learn how to jumpstart your bu…",2026-09-23 22:45:00 UTC,2026-11-12 01:15:00 UTC,"""official"""
2,30,"""Education Abroad: Returnee Pho…",null,2026-10-16 12:00:00 UTC,2026-11-20 21:00:00 UTC,"""official"""
3,32,"""Education Abroad: CIEE FDDO In…","""Getting Started""",2026-10-16 15:00:00 UTC,2026-12-16 17:00:00 UTC,"""official"""
4,32,"""Education Abroad: CIEE Scan De…","""The Scan Design Foundation Env…",2026-10-16 15:00:00 UTC,2026-11-17 16:00:00 UTC,"""official"""
5,36,"""IdeaSprint""","""Have an idea for how to solve …",2026-10-16 21:30:00 UTC,2026-11-18 17:00:00 UTC,"""official"""


4c. Lists are exploded: an event with 3 hosts becomes 3 event_organizer rows. For an event found on two sources, if one gave a real organizer and the other only the fallback (F7), we drop the fallback.

In [ ]:
def drop_fallback(links, name_col, fallback_names):
    # Keep a fallback value only if it's the event's only value
    is_fallback = pl.col(name_col).is_in(fallback_names)
    has_real = (~is_fallback).any().over("event_id")
    return links.filter(~(is_fallback & has_real))


event_organizer_links = drop_fallback( 
    rows_with_ids.select("event_id", pl.col("organizers").alias("organizer_name")).explode("organizer_name").unique(),
    "organizer_name", list(set(FALLBACK_ORGANIZER.values())),
)
event_category_links = drop_fallback(
    rows_with_ids.select("event_id", pl.col("categories").alias("category_name")).explode("category_name").unique(),
    "category_name", [FALLBACK_CATEGORY],
)

organizer = (
    event_organizer_links.select("organizer_name").unique().sort("organizer_name")
    .with_row_index("organizer_id", offset=1)
    .with_columns(
        pl.col("organizer_id").cast(pl.Int32),
        pl.lit(None, dtype=pl.String).alias("organizer_type"),   # no source has it
    )
)

category = (
    event_category_links.select("category_name").unique().sort("category_name")
    .with_row_index("category_id", offset=1)
    .with_columns(pl.col("category_id").cast(pl.Int32))
)

event_organizer = (
    event_organizer_links
    .join(organizer.select("organizer_id", "organizer_name"), on="organizer_name")
    .select("event_id", "organizer_id").unique().sort("event_id", "organizer_id")
    .with_row_index("event_organizer_id", offset=1)
    .with_columns(pl.col("event_organizer_id").cast(pl.Int32))
)

event_category = (
    event_category_links
    .join(category.select("category_id", "category_name"), on="category_name")
    .select("event_id", "category_id").unique().sort("event_id", "category_id")
    .with_row_index("event_category_id", offset=1)
    .with_columns(pl.col("event_category_id").cast(pl.Int32))
)


event_source = (
    rows_with_ids.join(source.select("source_id", "source_name"), on="source_name")
    .select("event_id", "source_id", "source_url")
    .unique(subset=["event_id", "source_id"], keep="first", maintain_order=True)
    .sort("event_id", "source_id")
    .with_row_index("event_source_id", offset=1)  
    .with_columns(
        pl.col("event_source_id").cast(pl.Int32),
        pl.lit(DATE_FOUND).alias("date_found"),   # when we downloaded it
    )
)

# Keep only the database's columns, in its order
building = building.select("building_id", "building_name", "latitude", "longitude")
location = location.select("location_id", "building_id", "room_number")

tables = {
    "building": building, "location": location, "organizer": organizer, "category": category,
    "event": event, "event_organizer": event_organizer, "event_category": event_category,
    "event_source": event_source,
}
for name, df in tables.items():
    print(f"{name:16} {df.height:4} rows   {df.columns}")

Rows with IDs:
Schema({'title': String, 'description': String, 'start_time': Datetime(time_unit='us', time_zone='UTC'), 'end_time': Datetime(time_unit='us', time_zone='UTC'), 'location': String, 'organizers': List(String), 'categories': List(String), 'source_name': String, 'source_url': String, 'building_name': String, 'room_number': String, 'official_status': String, 'event_key': String, 'priority': Int64, 'location_key': String, 'event_id': Int32})

Source:
Schema({'source_id': Int64, 'source_name': String, 'source_type': String})

Source values:
shape: (3, 3)
┌───────────┬─────────────────────────┬─────────────┐
│ source_id ┆ source_name             ┆ source_type │
│ ---       ┆ ---                     ┆ ---         │
│ i64       ┆ str                     ┆ str         │
╞═══════════╪═════════════════════════╪═════════════╡
│ 1         ┆ WildcatSync             ┆ platform    │
│ 2         ┆ Hurt Hub                ┆ website     │
│ 3         ┆ Davidson College Events ┆ website     │

/tmp/ipykernel_2980467/2492410961.py:9: DeprecationWarning: In Polars 2.0, the default behavior for `empty_as_null` will change to `False`. To keep the current behavior, explicitly set `empty_as_null=True`.
  rows_with_ids.select("event_id", pl.col("organizers").alias("organizer_name")).explode("organizer_name").unique(),
/tmp/ipykernel_2980467/2492410961.py:13: DeprecationWarning: In Polars 2.0, the default behavior for `empty_as_null` will change to `False`. To keep the current behavior, explicitly set `empty_as_null=True`.
  rows_with_ids.select("event_id", pl.col("categories").alias("category_name")).explode("category_name").unique(),


### 4d. Checks before loading
The database would reject bad rows anyway, but a failed check here is easier to debug than an error halfway through a load.
1. Primary keys: no duplicates, no nulls.
2. Unique rules: the same ones the database has.
3. Referential integrity: every foreign key exists in the table it points to.
4. At least one: every event has an organizer, a category and a source.
5. Nothing lost: joining the tables back together gives back every (event, source) pair.

In [63]:
def check(condition, message):
    assert condition, "FAILED: " + message
    print("ok  ", message)


# 1. Primary keys
for name, df in tables.items():
    pk = f"{name}_id"
    check(df[pk].null_count() == 0 and df[pk].is_unique().all(), f"{name}.{pk} unique and not null")

# 2. Unique rules from the schema
check(building["building_name"].is_unique().all(), "building_name unique")
check(not location.select("building_id", "room_number").is_duplicated().any(), "(building_id, room_number) unique")
check(organizer["organizer_name"].is_unique().all(), "organizer_name unique")
check(category["category_name"].is_unique().all(), "category_name unique")
for name, cols in (("event_organizer", ["event_id", "organizer_id"]),
                   ("event_category", ["event_id", "category_id"]),
                   ("event_source", ["event_id", "source_id"])):
    check(not tables[name].select(cols).is_duplicated().any(), f"{name} pair unique")

# 3. Referential integrity: (child table, column, parent table, column)
fks = [
    (location, "building_id", building, "building_id"),
    (event, "location_id", location, "location_id"),
    (event_organizer, "event_id", event, "event_id"),
    (event_organizer, "organizer_id", organizer, "organizer_id"),
    (event_category, "event_id", event, "event_id"),
    (event_category, "category_id", category, "category_id"),
    (event_source, "event_id", event, "event_id"),
    (event_source, "source_id", source, "source_id"),
]
for child, col, parent, pcol in fks:
    check(child[col].null_count() == 0 and child[col].is_in(parent[pcol].implode()).all(), f"{col} exists in parent")

# 4. Every event has at least one organizer, category and source
for bridge_name in ("event_organizer", "event_category", "event_source"):
    check(event["event_id"].is_in(tables[bridge_name]["event_id"].implode()).all(), f"every event has a row in {bridge_name}")

# The CHECK constraints in the schema
check(event.filter(pl.col("end_time") < pl.col("start_time")).height == 0, "end_time >= start_time")
check(event["official_status"].is_in(["official", "student_organized"]).all(), "official_status allowed")

ok   building.building_id unique and not null
ok   location.location_id unique and not null
ok   organizer.organizer_id unique and not null
ok   category.category_id unique and not null
ok   event.event_id unique and not null
ok   event_organizer.event_organizer_id unique and not null
ok   event_category.event_category_id unique and not null
ok   event_source.event_source_id unique and not null
ok   building_name unique
ok   (building_id, room_number) unique
ok   organizer_name unique
ok   category_name unique
ok   event_organizer pair unique
ok   event_category pair unique
ok   event_source pair unique
ok   building_id exists in parent
ok   location_id exists in parent
ok   event_id exists in parent
ok   organizer_id exists in parent
ok   event_id exists in parent
ok   category_id exists in parent
ok   event_id exists in parent
ok   source_id exists in parent
ok   every event has a row in event_organizer
ok   every event has a row in event_category
ok   every event has a row in event_

In [64]:
# 5. Rebuild every (event, source) pair from the tables and compare
rebuilt = (
    event_source.join(event, on="event_id").join(source, on="source_id")
    .join(location, on="location_id").join(building, on="building_id")
    .select("event_id", "source_name")
)
expected = rows_with_ids.select("event_id", "source_name").unique()
check(rebuilt.height == expected.height, f"every (event, source) pair survived ({rebuilt.height})")
check(rebuilt.join(expected, on=["event_id", "source_name"], how="anti").height == 0, "no extra (event, source) pairs")
check(event_organizer_links.height == event_organizer.height, f"every organizer link survived ({event_organizer.height})")
check(event_category_links.height == event_category.height, f"every category link survived ({event_category.height})")

ok   every (event, source) pair survived (153)
ok   no extra (event, source) pairs
ok   every organizer link survived (210)
ok   every category link survived (156)


## 5. Load
- **Parents first:** building → location → event → bridge tables.
- **One transaction:** everything is inside `engine.begin()`. If any insert fails, the whole load is undone.
- **Append only:** the migrations own the tables; we only insert rows.
- **Safe to rerun:** we empty the tables this notebook fills (`TRUNCATE ... RESTART IDENTITY`) and reload them. We never truncate `source`, because its rows come from a migration.

Because we truncate and reload, `created_at` means the time of the latest load.

Our IDs are `GENERATED ALWAYS AS IDENTITY`, which normally refuses IDs you supply. We insert the IDs we checked in section 4, so we use `OVERRIDING SYSTEM VALUE`. Afterwards we move each ID counter past the highest ID, so future inserts don't collide.

In [65]:
LOAD_ORDER = ["building", "location", "organizer", "category", "event",
              "event_organizer", "event_category", "event_source"]   # never includes source


def insert(conn, table, df):
    cols = ", ".join(df.columns)
    params = ", ".join(f":{c}" for c in df.columns)
    sql = text(f"INSERT INTO {table} ({cols}) OVERRIDING SYSTEM VALUE VALUES ({params})")
    conn.execute(sql, df.to_dicts())


with engine.begin() as conn:   # one transaction: all or nothing
    conn.execute(text(f"TRUNCATE {', '.join(LOAD_ORDER)} RESTART IDENTITY"))
    for table in LOAD_ORDER:
        insert(conn, table, tables[table])
        print(f"loaded {table:16} {tables[table].height:4} rows")
    for table in LOAD_ORDER:
        # move the ID counter past the highest ID we inserted
        conn.execute(text(
            f"SELECT setval(pg_get_serial_sequence('{table}', '{table}_id'), "
            f"COALESCE((SELECT MAX({table}_id) FROM {table}), 1))"
        ))
print("Load committed.")

loaded building           24 rows
loaded location           46 rows
loaded organizer          47 rows
loaded category           16 rows
loaded event             151 rows
loaded event_organizer   210 rows
loaded event_category    156 rows
loaded event_source      153 rows
Load committed.


### 5a. Verify
Each table's row count in the database must equal its DataFrame. Then one analyst-style query joining several tables.

In [66]:
with engine.connect() as conn:
    counts = []
    for table in LOAD_ORDER + ["source"]:
        in_db = conn.execute(text(f"SELECT COUNT(*) FROM {table}")).scalar_one()
        expected_rows = source.height if table == "source" else tables[table].height
        counts.append({"table": table, "dataframe": expected_rows, "database": in_db, "match": in_db == expected_rows})
counts = pl.DataFrame(counts)
assert counts["match"].all(), "row counts don't match"
counts

table,dataframe,database,match
str,i64,i64,bool
"""building""",24,24,true
"""location""",46,46,true
"""organizer""",47,47,true
"""category""",16,16,true
"""event""",151,151,true
"""event_organizer""",210,210,true
"""event_category""",156,156,true
"""event_source""",153,153,true
"""source""",3,3,true


In [67]:
query = """
SELECT c.category_name,
       s.source_name,
       COUNT(DISTINCT e.event_id)    AS events,
       COUNT(DISTINCT b.building_id) AS buildings
FROM event AS e
JOIN event_category AS ec ON ec.event_id = e.event_id
JOIN category       AS c  ON c.category_id = ec.category_id
JOIN event_source   AS es ON es.event_id = e.event_id
JOIN source         AS s  ON s.source_id = es.source_id
JOIN location       AS l  ON l.location_id = e.location_id
JOIN building       AS b  ON b.building_id = l.building_id
GROUP BY c.category_name, s.source_name
ORDER BY events DESC
LIMIT 15
"""
with engine.connect() as conn:
    result = pl.read_database(query, conn)
result

category_name,source_name,events,buildings
str,str,i64,i64
"""General""","""Davidson College Events""",43,7
"""Thoughtful Learning""","""WildcatSync""",39,7
"""Spirituality""","""WildcatSync""",15,4
"""Social""","""WildcatSync""",15,8
"""Athletics""","""WildcatSync""",15,2
…,…,…,…
"""Social""","""Davidson College Events""",2,2
"""Discussion""","""Hurt Hub""",2,2
"""Virtual""","""Hurt Hub""",1,1
